# 🗓️ 23일차 스터디 노트북 — 병합 정렬 (분할 정복의 완성)

**오늘 범위**: 06-7 병합 정렬 전체 — 실습 6-14(정렬된 두 배열 병합) · 조금만 더(`sorted(a+b)`, `heapq.merge()`) · 실습 6-15(병합 정렬) · 시간 복잡도와 안정성

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** 손으로 추적 · **[빈칸]** 빈칸 채우기 · **[예측]** 실행 전 결과 맞히기 · **[구현]** · **[디버깅]** · **[설명]** · **[실험]** · **[비교]**

---

## 오늘의 세 가지 질문

> **Q1. 퀵 정렬은 "나누고 나서 정렬"인데, 병합 정렬은 "정렬하고 나서 합친다". 이 순서 차이가 뭘 바꿀까?**
>
> **Q2. 배열 앞부분만 `buff`에 복사하고 뒷부분은 안 하는데, 왜 원본이 안 깨질까?** 🔥
>
> **Q3. 퀵 정렬보다 느린데, 왜 파이썬 `sorted()`는 병합 정렬 계열을 골랐을까?**

오늘은 **버그가 없는 날**이야. 그래서 문제도 "틀린 곳 찾기"보다 **"왜 이렇게 생겼는지 끝까지 따지기"** 쪽에 무게를 뒀어.

> 📁 아래 **"부록: 오늘의 코드 모음"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 코드 모음 (제일 먼저 실행!)

In [ ]:
from typing import Sequence, MutableSequence
import sys, random, time, math, heapq, tracemalloc
sys.setrecursionlimit(200000)

# ---------- 실습 6-14: 정렬을 마친 두 배열의 병합 ----------
def merge_sorted_list(a: Sequence, b: Sequence, c: MutableSequence) -> None:
    """정렬을 마친 배열 a와 b를 병합하여 c에 저장"""
    pa, pb, pc = 0, 0, 0
    na, nb, nc = len(a), len(b), len(c)

    while pa < na and pb < nb:
        if a[pa] <= b[pb]:
            c[pc] = a[pa]; pa += 1
        else:
            c[pc] = b[pb]; pb += 1
        pc += 1

    while pa < na:
        c[pc] = a[pa]; pa += 1; pc += 1

    while pb < nb:
        c[pc] = b[pb]; pb += 1; pc += 1


# ---------- 실습 6-15: 병합 정렬 ----------
def merge_sort(a: MutableSequence) -> None:
    """병합 정렬"""
    def _merge_sort(a: MutableSequence, left: int, right: int) -> None:
        """a[left] ~ a[right]를 재귀적으로 병합 정렬"""
        if left < right:
            center = (left + right) // 2

            _merge_sort(a, left, center)        # 앞부분
            _merge_sort(a, center + 1, right)   # 뒷부분

            p = j = 0
            i = k = left

            while i <= center:                  # ⓐ 앞부분을 buff로 복사
                buff[p] = a[i]; p += 1; i += 1

            while i <= right and j < p:         # ⓑ buff와 뒷부분을 병합
                if buff[j] <= a[i]:
                    a[k] = buff[j]; j += 1
                else:
                    a[k] = a[i]; i += 1
                k += 1

            while j < p:                        # ⓒ buff에 남은 것 복사
                a[k] = buff[j]; k += 1; j += 1

    n = len(a)
    buff = [None] * n
    _merge_sort(a, 0, n - 1)
    del buff


# ---------- 21일차 퀵 정렬 (비교용) ----------
def quick_sort(a: MutableSequence) -> None:
    def qsort(a, left, right):
        pl, pr = left, right
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
        if left < pr:  qsort(a, left, pr)
        if pl < right: qsort(a, pl, right)
    if len(a) > 0:
        qsort(a, 0, len(a) - 1)

print("준비 완료 ✅")

---
# 🔁 [Remind] 워밍업 — 22일차 되감기

어제 만든 두 개가 오늘 그대로 쓰여. 하나는 **부품으로**, 하나는 **대조군으로**.

### R-1. 🟢 [손] 어제 만든 `merge_sorted_list`

22일차 17번에서 짠 그 함수야. 아래 입력에 대해 **`c`가 채워지는 순서**를 손으로 적어봐.

```
a = [2, 5, 5, 9]
b = [1, 5, 7]
```

| 단계 | 비교 | 꺼낸 값 | 어느 배열에서 | c |
|---|---|---|---|---|
| 1 | `2 <= 1`? | | | |
| 2 | | | | |
| … | | | | |

- 값이 **5로 같을 때**(a의 5 vs b의 5) 어느 쪽이 먼저 나가? 그 이유가 되는 **연산자 한 글자**는?
- 마지막에 실행되는 while은 ③번이야, ④번이야?

### R-2. 🟡 [설명] 어제의 대조군 — 퀵 정렬

22일차 6~7번에서 비재귀 퀵 정렬의 **스택 최대 크기**가 push 순서에 따라 log n 이하로 눌린다는 걸 봤지.

- 퀵 정렬은 **"나눈 다음에 각각 정렬"** 이야. 그럼 나누는 기준(피벗)이 나쁘면 어떻게 되지?
- 오늘 배울 병합 정렬은 **"각각 정렬한 다음에 합친다"**. 나누는 기준이 `(left+right)//2` 로 **고정**이야.
- 그럼 병합 정렬의 재귀 깊이는 **입력에 따라 달라질까, 달라지지 않을까?** 예측해두고 15번에서 확인해.

*(여기에 답 작성)*

In [ ]:
a = [2, 5, 5, 9]
b = [1, 5, 7]
c = [None] * (len(a) + len(b))
merge_sorted_list(a, b, c)
print("c =", c)

---
# 🧩 PART 1 — 병합이라는 부품 (1~6번)

> 병합 정렬을 이해하는 순서는 **"합치기 → 쪼개기"** 야. 부품부터.

### 1. 🟢 [손] 세 개의 커서

교재 [그림 6-27] (277p) 그대로야.

```
a = [2, 4, 6, 8, 11, 13]
b = [1, 2, 3, 4, 9, 16, 21]
```

`pa`, `pb`, `pc` 세 커서가 있어. **처음 5번의 `pc` 증가**까지만 표를 채워봐.

| pc | 비교 `a[pa] <= b[pb]` | 저장된 값 | 이후 (pa, pb) |
|---|---|---|---|
| 0 | `2 <= 1`? 거짓 | 1 | (0, 1) |
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |

- 값이 **2로 같은 순간**이 한 번 나와. 그때 어느 쪽이 먼저 나가지?
- 커서 3개 중 **매 반복마다 반드시 1 증가하는 것**은 뭐야? 나머지 둘은?

*(답을 적은 뒤 아래 셀로 확인)*

In [ ]:
def merge_trace(a, b):
    """merge_sorted_list + 매 단계 출력"""
    c = [None] * (len(a) + len(b))
    pa = pb = pc = 0
    na, nb = len(a), len(b)
    while pa < na and pb < nb:
        if a[pa] <= b[pb]:
            print(f"pc={pc} | a[{pa}]={a[pa]} <= b[{pb}]={b[pb]} → a에서 {a[pa]} 꺼냄")
            c[pc] = a[pa]; pa += 1
        else:
            print(f"pc={pc} | a[{pa}]={a[pa]} <= b[{pb}]={b[pb]} 거짓 → b에서 {b[pb]} 꺼냄")
            c[pc] = b[pb]; pb += 1
        pc += 1
    while pa < na:
        print(f"pc={pc} | ③번 while: a에 남은 {a[pa]}")
        c[pc] = a[pa]; pa += 1; pc += 1
    while pb < nb:
        print(f"pc={pc} | ④번 while: b에 남은 {b[pb]}")
        c[pc] = b[pb]; pb += 1; pc += 1
    return c

print(merge_trace([2, 4, 6, 8, 11, 13], [1, 2, 3, 4, 9, 16, 21]))

### 2. 🟢 [설명] 왜 `while`이 세 개일까

```python
while pa < na and pb < nb:   # ①
    ...
while pa < na:               # ②
    ...
while pb < nb:               # ③
```

- ①이 끝났다는 건 무슨 뜻이지? (두 조건 중 **적어도 하나**가 깨졌다는 것)
- 그렇다면 ②와 ③은 **동시에 실행될 수 있을까?** 왜?
- ②와 ③을 둘 다 지우면 어떤 입력에서 문제가 생겨? 구체적인 예를 하나 만들어봐.
- 총 `c`에 저장되는 횟수는 정확히 몇 번이야? 그래서 시간 복잡도가 **O(n)** 인 거지. 교재 277p가 말하는 그거야.

*(여기에 답 작성)*

### 3. 🟡 [예측] ②번 while을 지우면

`while pa < na:` 블록만 통째로 지웠다고 하자.

```
a = [1, 2, 9]
b = [3, 4]
```

- 이 입력에서 결과 `c`는 어떻게 될까? **실행하기 전에** 예측해봐.
- 에러가 날까, 아니면 조용히 틀린 답이 나올까?
- 반대로 `a = [1, 2]`, `b = [3, 4, 9]` 라면?

💡 22일차 10번에서 배운 것 — **"조용히 틀리는 것"이 더 무섭다.**

*(예측을 적은 뒤 실행)*

In [ ]:
def merge_no_tail_a(a, b):
    c = [None] * (len(a) + len(b))
    pa = pb = pc = 0
    while pa < len(a) and pb < len(b):
        if a[pa] <= b[pb]: c[pc] = a[pa]; pa += 1
        else:              c[pc] = b[pb]; pb += 1
        pc += 1
    # while pa < len(a): ...  ← 통째로 삭제
    while pb < len(b):
        c[pc] = b[pb]; pb += 1; pc += 1
    return c

print("a=[1,2,9], b=[3,4] →", merge_no_tail_a([1, 2, 9], [3, 4]))
print("a=[1,2],   b=[3,4,9] →", merge_no_tail_a([1, 2], [3, 4, 9]))

### 4. 🟡 [비교] 병합하는 세 가지 방법

교재 279p "조금만 더"에 나온 대안들이야. 노트에도 `c = list(sorted(a+b))` 를 크게 적어놨지.

| 방법 | 코드 | 시간 복잡도 | a, b가 정렬 안 돼 있어도 OK? |
|---|---|---|---|
| 직접 병합 | `merge_sorted_list(a, b, c)` | ① | ② |
| 이어붙여 정렬 | `c = list(sorted(a + b))` | ③ | ④ |
| heapq | `c = list(heapq.merge(a, b))` | ⑤ | ⑥ |

- 표를 채워봐.
- 교재는 `sorted(a+b)` 에 대해 **"적용 범위는 넓지만 속도가 빠르지 않다"** 고 해. 왜 느릴까? (이미 정렬된 정보를 **버리고** 처음부터 다시 하니까)
- 그럼 `heapq.merge()`는 왜 빠를까? 이름에 답이 있어 — 힙을 써서 **각 배열의 맨 앞만 비교**하거든. (힙은 06-8에서 배울 예정!)

*(표를 채운 뒤 아래 셀로 속도 확인)*

In [ ]:
random.seed(1)
N = 200000
a = sorted(random.randint(0, 10**6) for _ in range(N))
b = sorted(random.randint(0, 10**6) for _ in range(N))

t = time.perf_counter()
c1 = [None] * (2 * N); merge_sorted_list(a, b, c1)
t1 = time.perf_counter() - t

t = time.perf_counter()
c2 = list(sorted(a + b))
t2 = time.perf_counter() - t

t = time.perf_counter()
c3 = list(heapq.merge(a, b))
t3 = time.perf_counter() - t

print(f"직접 병합(파이썬 루프) : {t1:6.3f}s")
print(f"sorted(a + b)          : {t2:6.3f}s")
print(f"heapq.merge(a, b)      : {t3:6.3f}s")
print("세 결과 일치:", c1 == c2 == c3)

### 5. 🟡 [실험] `heapq.merge()`는 리스트가 아니다

교재 279p 코드는 `c = list(heapq.merge(a, b))` 처럼 **`list()`로 감싸**. 왜일까?

- `heapq.merge(a, b)`를 `list()` 없이 그냥 출력하면 뭐가 나올까? 예측해봐.
- 이걸 **두 번** 순회하면 어떻게 될까?
- 💡 이 성질(한 번만 흘러가는 것)이 오히려 **장점**이 되는 상황은 언제일까? (힌트: 100만 개짜리 배열 10개를 병합해서 **앞 10개만** 필요하다면?)

*(예측을 적은 뒤 실행)*

In [ ]:
a = [2, 4, 6, 8, 11, 13]
b = [1, 2, 3, 4, 9, 16, 21]

m = heapq.merge(a, b)
print("타입:", type(m).__name__)
print("그냥 출력:", m)
print("list()로 감싸면:", list(heapq.merge(a, b)))

g = heapq.merge(a, b)
first = list(g)
second = list(g)
print(f"\n같은 객체를 두 번 순회 → 1회차 {len(first)}개, 2회차 {len(second)}개")

# 앞 5개만 필요하다면? — 200만 개를 다 계산하지 않는다
big1 = list(range(0, 2000000, 2))
big2 = list(range(1, 2000000, 2))

t = time.perf_counter()
gen = heapq.merge(big1, big2)
top5 = [next(gen) for _ in range(5)]
t_lazy = time.perf_counter() - t

t = time.perf_counter()
top5_sorted = sorted(big1 + big2)[:5]
t_eager = time.perf_counter() - t

print(f"\nheapq.merge로 앞 5개만 : {top5} ({t_lazy:.5f}s)")
print(f"sorted(a+b)[:5]        : {top5_sorted} ({t_eager:.5f}s)")
print(f"→ {t_eager/t_lazy:.0f}배 차이")


### 6. 🟢 [손] 쪼개기 — 분할 정복의 그림

교재 [그림 6-28] (280p). 원소 12개짜리 배열이야.

```
[5, 6, 4, 8, 3, 7, 9, 0, 1, 5, 2, 3]
```

- `left=0`, `right=11` → `center = ①____`
- 앞부분은 `a[0]~a[②____]`, 뒷부분은 `a[③____]~a[11]`
- 이 배열이 **원소 1개짜리가 될 때까지** 쪼개지는 데 몇 단계가 필요해? (12 → 6 → 3 → 2 → 1)
- 원소가 n개일 때 이 단계 수는 대략 **④____** 야.

**교재 280p의 알고리즘 3줄**을 그대로 적어봐:
```
배열의 원소 수가 2개 이상인 경우
1. ⑤________________
2. ⑥________________
3. ⑦________________
```

- 이 3줄에서 **재귀 호출**은 몇 번 등장해? 21일차 퀵 정렬의 마지막 두 줄과 비교하면 구조가 같아 보이는데, **결정적으로 다른 점**은 뭘까? (힌트: "정렬"이 언제 일어나는가)

*(답을 적은 뒤 아래 셀로 분할 과정 확인)*

In [ ]:
def show_split(left, right, depth=0):
    """분할 과정을 트리로 출력"""
    print("  " * depth + f"[{left}..{right}] (원소 {right-left+1}개)")
    if left < right:
        center = (left + right) // 2
        show_split(left, center, depth + 1)
        show_split(center + 1, right, depth + 1)

show_split(0, 11)

---
# 🔬 PART 2 — `merge_sort` 해부 (7~13번)

> 실습 6-15는 20줄 남짓인데, 그 안에 **왜 그렇게 써야만 하는지**가 빽빽하게 들어 있어.
> 특히 9번은 오늘의 핵심이야.

### 7. 🟢 [설명] `if left < right:` — 기저 조건

```python
def _merge_sort(a, left, right):
    if left < right:
        ...
```

- 이 조건이 **거짓**이 되는 건 언제야? `left == right`는 원소가 몇 개라는 뜻이지?
- 원소 1개짜리 배열은 이미 정렬되어 있으니 아무것도 안 해도 돼. 그래서 **그냥 리턴**하는 거야.
- 만약 `if left <= right:` 로 바꾸면 무슨 일이 생길까? (`left == right`일 때 `center`는 뭐가 되고, `_merge_sort(a, left, center)`는 어떤 인자로 호출되지?)
- 22일차 3번에서 봤던 **"기저 조건이 없으면 무한히 자란다"** 와 같은 이야기야. 그때는 스택이 폭발했고, 지금은 뭐가 폭발할까?

*(답을 적은 뒤 실행 — 안전장치 걸려 있음)*

In [ ]:
def merge_sort_le(a):
    """if left <= right 로 바꾼 버전"""
    def _ms(a, left, right, depth=0):
        if depth > 60:
            raise RecursionError(f"깊이 {depth} 초과 — left={left}, right={right}")
        if left <= right:                 # 🐛 <= 로 변경
            center = (left + right) // 2
            _ms(a, left, center, depth + 1)
            _ms(a, center + 1, right, depth + 1)
            p = j = 0; i = k = left
            while i <= center: buff[p] = a[i]; p += 1; i += 1
            while i <= right and j < p:
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None] * n
    _ms(a, 0, n - 1)

try:
    merge_sort_le([5, 8, 4, 2])
except RecursionError as e:
    print("RecursionError:", e)

### 8. 🟢 [손] `buff`를 쓰는 3단계

교재 [그림 6-31] (284p). `left=0`, `center=5`, `right=11` 이고 앞뒤가 각각 정렬을 마친 상태야.

```
a = [1, 3, 4, 6, 11, 12, | 2, 3, 5, 6, 7, 8]
         앞부분(정렬됨)      뒷부분(정렬됨)
```

**ⓐ 단계** (19~22행)
- `while i <= center:` 가 끝났을 때 `buff`에 담긴 것은? ①____
- 이때 `p`의 값은? ②____ (= `center - left + 1`)

**ⓑ 단계** (24~31행)
- `buff`와 `a[6]~a[11]`을 비교하며 `a[k]`에 채워넣어. 처음 4번의 `(k, 저장값)`을 적어봐.
- 이 while이 끝나는 두 가지 경우는? ③____ 또는 ④____

**ⓒ 단계** (33~36행)
- 여기서 복사되는 건 `buff`에 남은 것뿐이야. **`a`의 뒷부분이 남았을 때를 위한 while은 왜 없을까?** 🔥
  💡 힌트: 뒷부분이 남았다면, 그건 이미 `a`의 **제자리에** 있는 거잖아?

*(답을 적은 뒤 실행)*

In [ ]:
a = [1, 3, 4, 6, 11, 12, 2, 3, 5, 6, 7, 8]
left, center, right = 0, 5, 11
buff = [None] * len(a)

p = j = 0
i = k = left
while i <= center:
    buff[p] = a[i]; p += 1; i += 1
print(f"ⓐ buff = {buff[:p]},  p = {p}")

print("ⓑ 병합:")
while i <= right and j < p:
    if buff[j] <= a[i]:
        print(f"   k={k} ← buff[{j}]={buff[j]}")
        a[k] = buff[j]; j += 1
    else:
        print(f"   k={k} ← a[{i}]={a[i]}")
        a[k] = a[i]; i += 1
    k += 1

print(f"ⓒ buff에 남은 것 복사 (j={j}, p={p}):")
while j < p:
    print(f"   k={k} ← buff[{j}]={buff[j]}")
    a[k] = buff[j]; k += 1; j += 1

print("\n결과:", a)

### 9. 🔴 [설명] 🔥 오늘의 핵심 — 왜 앞부분만 복사할까

```python
while i <= center:          # 앞부분만 buff로 복사
    buff[p] = a[i]; p += 1; i += 1
```

병합하려면 배열이 두 개 필요한데, 교재는 **앞부분만** `buff`에 떠놓고 **뒷부분은 `a`에 그대로 둔 채** 병합해. 그런데 결과도 `a`에 덮어써.

**당연히 이런 의심이 들어야 해**: 결과를 `a[k]`에 쓰는데, 아직 안 읽은 `a[i]`를 덮어쓰면 어떡하지?

- 결과를 쓰는 위치 `k`는 `left`에서 시작해서 매번 1씩 증가해.
- 뒷부분을 읽는 위치 `i`는 `center + 1`에서 시작해.
- 그럼 **`k`가 `i`를 따라잡을 수 있을까?** 아래 부등식을 채워서 확인해봐:

```
k가 지금까지 쓴 개수  =  (buff에서 꺼낸 개수 j) + (a 뒷부분에서 꺼낸 개수 i - center - 1)
따라서  k = left + j + (i - center - 1)
그런데  j <= p = center - left + 1  이므로
        k <= left + (center - left + 1) + (i - center - 1) = ①____
```

- ①에 뭐가 나와? 그래서 결론은 **`k ≤ ②____`** 이고, 이건 "쓰는 위치가 읽는 위치를 **절대 추월하지 않는다**"는 뜻이야.
- 그럼 **앞부분은 왜 복사해야만 할까?** `k`가 `left`부터 시작하는데 앞부분 원본도 `left`부터 있으니, 복사 안 하면 **첫 쓰기부터 바로** 덮어써버리거든.
- 💡 이런 걸 **불변식(invariant)** 이라고 해. 22일차 14번에서 `insertion_sort`의 `j > 0`이 "우연히 안전"했던 것과 달리, 이건 **의도적으로 설계된 안전성**이야. 차이를 설명해봐.

*(답을 적은 뒤 실행 — 실제로 `k <= i`가 유지되는지 검증)*

In [ ]:
viol = [0]
samples = []

def merge_sort_check(a):
    def _ms(a, left, right):
        if left < right:
            center = (left + right) // 2
            _ms(a, left, center); _ms(a, center + 1, right)
            p = j = 0; i = k = left
            while i <= center: buff[p] = a[i]; p += 1; i += 1
            while i <= right and j < p:
                if k > i: viol[0] += 1
                if len(samples) < 8 and right - left > 6:
                    samples.append((k, i, k <= i))
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None] * n; _ms(a, 0, n - 1)

random.seed(0)
for _ in range(500):
    t = [random.randint(0, 50) for _ in range(random.randint(1, 60))]
    merge_sort_check(t)

print(f"k > i 위반 횟수: {viol[0]}  ← 0이면 불변식 성립")
print("\n샘플 (k, i, k<=i):")
for s in samples:
    print("  ", s)

### 10. 🟡 [디버깅] `and j < p` 를 지우면

```python
while i <= right and j < p:
```

두 번째 조건 `j < p`를 지웠다고 하자.

- `j`가 `p`에 도달했다는 건 무슨 뜻이야? (`buff`에서 **다 꺼냈다**)
- 그 상태에서 `buff[j]`를 읽으면 뭐가 나올까? `buff`는 `[None] * n`으로 만들었잖아.
- 그럼 어떤 에러가 날까? **에러 이름과 메시지를 예측**해봐.
- 반대로 `i <= right` 조건만 지우면? (이번엔 예측만, 실행은 각자)

*(예측을 적은 뒤 실행)*

In [ ]:
def merge_sort_broken(a):
    def _ms(a, left, right):
        if left < right:
            center = (left + right) // 2
            _ms(a, left, center); _ms(a, center + 1, right)
            p = j = 0; i = k = left
            while i <= center: buff[p] = a[i]; p += 1; i += 1
            while i <= right:                 # 🐛 and j < p 제거
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None] * n; _ms(a, 0, n - 1)

try:
    t = [5, 8, 4, 2, 6, 1, 3, 9, 7]
    merge_sort_broken(t)
    print("결과:", t)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

### 11. 🟡 [빈칸] 병합 정렬 완성

7~10번에서 뜯어본 걸 이제 직접 조립해.

**기대 출력**
```
[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def my_merge_sort(a: MutableSequence) -> None:
    """병합 정렬"""

    def _ms(a: MutableSequence, left: int, right: int) -> None:
        if ___:                              # ① 기저 조건
            center = ___                     # ② 중앙 인덱스

            _ms(a, ___, ___)                 # ③ 앞부분
            _ms(a, ___, ___)                 # ④ 뒷부분

            p = j = 0
            i = k = left

            while ___:                       # ⑤ 앞부분을 buff로
                buff[p] = a[i]; p += 1; i += 1

            while ___ and ___:               # ⑥ 병합 조건 두 개
                if ___:                      # ⑦ buff 쪽을 꺼낼 조건 (안정성!)
                    a[k] = buff[j]; j += 1
                else:
                    a[k] = a[i]; i += 1
                k += 1

            while ___:                       # ⑧ buff에 남은 것
                a[k] = buff[j]; k += 1; j += 1

    n = len(a)
    buff = [None] * n
    _ms(a, 0, n - 1)


x = [5, 8, 4, 2, 6, 1, 3, 9, 7, 0, 3, 5]
my_merge_sort(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 60)
    t = [random.randint(0, 40) for _ in range(n)]
    ref = sorted(t)
    my_merge_sort(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

### 12. 🔴 [실험] 🔥 `buff`를 재귀마다 만들면 진짜 손해일까

교재는 `buff`를 **`_merge_sort` 바깥에서 딱 한 번** 만들어.

```python
n = len(a)
buff = [None] * n        # ← 여기, 한 번만
_merge_sort(a, 0, n - 1)
```

만약 재귀 호출 안에서 매번 `buff = [None] * (center - left + 1)` 로 필요한 만큼만 만든다면?

**먼저 예측해봐** (n = 50,000):

| 항목 | 예측 |
|---|---|
| 실행 시간 | ① 어느 쪽이 빠를까? |
| 피크 메모리 | ② 어느 쪽이 적게 쓸까? |

그리고 실행해봐. **아마 예측이 틀릴 거야.** 틀렸다면, 왜 그런지 생각해보고 다음 질문에 답해:

- 교재가 밖에 둔 이유가 **성능** 때문이 아니라면, 진짜 이유는 뭘까? (힌트: 코드에서 사라지는 줄이 몇 줄인지, 그리고 크기 계산 실수 가능성)
- 💡 이 문제의 진짜 교훈: **"최적화처럼 보이는 것"이 실제로 최적화인지는 재봐야 안다.**

*(예측을 적은 뒤 실행)*

In [ ]:
def ms_outer(a):
    """buff를 한 번만 생성 (교재 방식)"""
    def _ms(a, l, r):
        if l < r:
            c = (l + r) // 2
            _ms(a, l, c); _ms(a, c + 1, r)
            p = j = 0; i = k = l
            while i <= c: buff[p] = a[i]; p += 1; i += 1
            while i <= r and j < p:
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None] * n; _ms(a, 0, n - 1); del buff

def ms_inner(a):
    """buff를 재귀마다 필요한 크기로 생성"""
    def _ms(a, l, r):
        if l < r:
            c = (l + r) // 2
            _ms(a, l, c); _ms(a, c + 1, r)
            buff = [None] * (c - l + 1)      # ← 매번 생성
            p = j = 0; i = k = l
            while i <= c: buff[p] = a[i]; p += 1; i += 1
            while i <= r and j < p:
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    _ms(a, 0, len(a) - 1)

random.seed(5)
data = [random.randint(0, 10**6) for _ in range(50000)]
ref = sorted(data)

for name, f in (("buff 1회 생성 (교재)", ms_outer), ("buff 매번 생성    ", ms_inner)):
    a = data[:]
    t = time.perf_counter(); f(a); el = time.perf_counter() - t
    a2 = data[:]
    tracemalloc.start(); f(a2); cur, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    print(f"{name}: {el:5.3f}s | 피크 메모리 {peak/1024:7.1f} KB | 정렬 {'OK' if a == ref else 'X'}")

### 13. 🟡 [설명] `buff`는 어디에 살고, `del buff`는 뭘 할까

두 가지를 따져보자.

**(1) 스코프**
```python
def merge_sort(a):
    def _merge_sort(a, left, right):
        ...
        buff[p] = a[i]          # ← buff를 여기서 쓰는데
    n = len(a)
    buff = [None] * n           # ← 정의는 여기, _merge_sort보다 '아래'
    _merge_sort(a, 0, n - 1)
```
- `_merge_sort`가 **정의될 때** `buff`는 아직 존재하지 않아. 그런데 왜 에러가 안 날까?
- 이런 걸 **클로저(closure)** 라고 해. 안쪽 함수가 바깥 함수의 변수를 참조하는 구조야. 이름이 언제 찾아지는지가 핵심 — **정의 시점**일까 **실행 시점**일까?

**(2) `del buff`**
```python
    _merge_sort(a, 0, n - 1)
    del buff                    # 작업용 배열을 소멸
```
- `del buff`는 정확히 무슨 일을 하지? 리스트의 **메모리를 즉시 해제**하는 걸까, 아니면 **이름표를 떼는** 걸까?
- `merge_sort` 함수가 어차피 곧 끝나는데, 이 줄이 있는 것과 없는 것의 실질적 차이는? (거의 없다면, 교재는 왜 썼을까?)

*(답을 적은 뒤 실행)*

In [ ]:
# (1) 클로저 — 이름은 실행 시점에 찾는다
def demo():
    def inner():
        return msg              # 아직 정의 안 된 이름
    msg = "실행 시점에 찾아짐!"
    return inner()
print("클로저:", demo())

def demo_fail():
    def inner():
        return not_defined_anywhere
    return inner()
try:
    demo_fail()
except NameError as e:
    print("NameError:", e)

# (2) del 은 이름표를 뗀다
lst = [1, 2, 3]
alias = lst                     # 같은 객체를 가리키는 두 번째 이름
del lst
print("\ndel lst 후 alias:", alias, "← 객체는 살아 있다")
try:
    lst
except NameError as e:
    print("NameError:", e)

---
# 📐 PART 3 — 성질과 확장 (14~17번)

> 교재 285p의 마지막 두 문장이 오늘의 결론이야.
> **"전체 시간 복잡도는 O(n log n)입니다."** / **"서로 떨어져 있는 원소를 교환하는 것이 아니므로 안정적입니다."**
> 이 두 문장을 각각 증명해보자.

### 14. 🟡 [실험] 안정성 — 어제 예고한 그 한 글자

22일차 18번에서 예고했지. 이제 **완성된 병합 정렬**로 확인해.

```python
if buff[j] <= a[i]:      # ← 이 <= 를 < 로 바꾸면?
```

입력 (값, 라벨):
```
[(3,'a'), (1,'b'), (3,'c'), (1,'d'), (2,'e'), (3,'f')]
```

- 안정 정렬이라면 결과 라벨 순서는 ①____ 이어야 해. (같은 값끼리 원래 순서 유지)
- `<` 로 바꾸면 어떻게 될까? 예측해봐.
- 교재 285p는 **"서로 떨어져 있는 원소를 교환하는 것이 아니므로 안정적"** 이라고 해. 이 표현을 21일차 퀵 정렬과 대조해서 설명해봐 — 퀵은 뭘 하길래 불안정하지?
- 마지막: `buff`가 **앞부분**을 담고 있다는 사실이 안정성에 왜 중요할까? (같은 값일 때 `buff` 쪽을 먼저 꺼낸다 = 원래 **앞쪽**에 있던 걸 먼저 꺼낸다)

*(답을 적은 뒤 실행)*

In [ ]:
def ms_stable(a, strict=False):
    def _ms(a, l, r):
        if l < r:
            c = (l + r) // 2
            _ms(a, l, c); _ms(a, c + 1, r)
            p = j = 0; i = k = l
            while i <= c: buff[p] = a[i]; p += 1; i += 1
            while i <= r and j < p:
                take = (buff[j][0] < a[i][0]) if strict else (buff[j][0] <= a[i][0])
                if take: a[k] = buff[j]; j += 1
                else:    a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None] * n; _ms(a, 0, n - 1)

base = [(3,'a'), (1,'b'), (3,'c'), (1,'d'), (2,'e'), (3,'f')]
print("입력       :", [x[1] for x in base], [x[0] for x in base])
for strict, label in ((False, "<= (교재)"), (True, "<  (바꿈)")):
    t = base[:]; ms_stable(t, strict)
    print(f"{label} :", [x[1] for x in t], [x[0] for x in t])
print("\n파이썬 sorted():", [x[1] for x in sorted(base, key=lambda t: t[0])])

### 15. 🟡 [실험] O(n log n)의 두 조각, 그리고 퀵과의 대결

교재 285p: **"배열 병합의 시간 복잡도는 O(n)입니다. 데이터 원소 수가 n일 때 병합 정렬의 단계는 log n 만큼 필요하므로 전체 시간 복잡도는 O(n log n)입니다."**

- **O(n)** 은 어디서 나왔지? (2번 문제)
- **log n** 은 어디서 나왔지? (6번 문제)
- R-2에서 예측한 것: 병합 정렬의 재귀 깊이는 입력에 따라 **달라질까?** 이제 확인해.

그리고 21일차 퀵 정렬과 비교해보자. 예측부터:

| 항목 | 병합 | 퀵 | 예측 |
|---|---|---|---|
| 랜덤 입력 속도 | | | ① |
| **이미 정렬된** 입력 속도 | | | ② |
| 추가 메모리 | | | ③ |
| 최악 시간 복잡도 | | | ④ |

*(예측을 적은 뒤 실행)*

In [ ]:
D = [0]
def ms_depth(a):
    def _ms(a, l, r, d=1):
        D[0] = max(D[0], d)
        if l < r:
            c = (l + r) // 2
            _ms(a, l, c, d+1); _ms(a, c+1, r, d+1)
            p = j = 0; i = k = l
            while i <= c: buff[p] = a[i]; p += 1; i += 1
            while i <= r and j < p:
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None] * n; _ms(a, 0, n - 1)

random.seed(3)
N = 100000
data = [random.randint(0, 10**6) for _ in range(N)]

print(f"[재귀 깊이]  log2({N}) = {math.log2(N):.1f}")
for name, d in (("랜덤 입력      ", data), ("이미 정렬된 입력", sorted(data)), ("역순 입력      ", sorted(data, reverse=True))):
    D[0] = 0; ms_depth(d[:])
    print(f"  {name}: 깊이 {D[0]}")

print("\n[속도 비교]")
for name, dset in (("랜덤 입력      ", data), ("이미 정렬된 입력", sorted(data))):
    row = []
    for algo, f in (("병합", merge_sort), ("퀵", quick_sort), ("내장", lambda x: x.sort())):
        a = dset[:]
        t = time.perf_counter(); f(a); row.append(f"{algo} {time.perf_counter()-t:.3f}s")
    print(f"  {name}: " + " | ".join(row))

### 16. 🔴 [구현] 재귀 없는 병합 정렬 (상향식)

22일차에 퀵 정렬을 **비재귀**로 만들었지. 병합 정렬도 가능해. 그런데 방식이 완전히 달라.

- 퀵 정렬은 "나눌 범위"를 **스택에 저장**해야 했어. 나누는 위치가 데이터에 따라 달라지니까.
- 병합 정렬은 나누는 위치가 **처음부터 정해져 있어** (항상 절반). 그래서 스택이 필요 없어!
- 대신 **아래에서 위로** 올라가면 돼. 크기 1짜리끼리 병합 → 크기 2끼리 병합 → 크기 4끼리 → …

```
[5, 8, 4, 2, 6, 1, 3, 9, 7]
width=1: [5,8] [4,2] [6,1] [3,9] [7]   → 둘씩 병합
width=2: [5,8,2,4] [1,6,3,9] [7]       → 넷씩 병합
width=4: [1,2,3,4,5,6,8,9] [7]
width=8: 전체 병합
```

**기대 출력**
```
[1, 2, 3, 4, 5, 6, 7, 8, 9]
랜덤 500회 검증: 실패 0회
재귀 깊이: 0 (재귀 없음)
```

**힌트**: `width`를 1부터 2배씩 키우면서, 시작 위치 `lo`를 `2*width`씩 건너뛰며 순회해. 마지막 조각이 잘릴 수 있으니 `min()`으로 경계를 막아야 해.

In [ ]:
def merge_sort_bottomup(a: MutableSequence) -> None:
    """재귀 없는 상향식 병합 정렬"""
    n = len(a)
    buff = [None] * n
    width = 1
    while width < n:
        for lo in range(0, n, ___):              # ① 몇 칸씩 건너뛸까?
            mid = min(lo + width, n)             # 앞 조각의 끝(배타적)
            hi  = min(lo + ___, n)               # ② 뒤 조각의 끝(배타적)
            if mid >= hi:
                continue                         # 뒤 조각이 없으면 병합 불필요

            # --- 앞 조각을 buff로 복사 ---
            p = 0; i = lo
            while i < mid:
                buff[p] = a[i]; p += 1; i += 1

            # --- buff와 뒤 조각을 병합 ---
            j = 0; k = lo; i = mid
            while i < hi and j < p:
                if ___:                          # ③ 안정성 유지!
                    a[k] = buff[j]; j += 1
                else:
                    a[k] = a[i]; i += 1
                k += 1
            while j < p:
                a[k] = buff[j]; k += 1; j += 1

        width *= ___                             # ④


x = [5, 8, 4, 2, 6, 1, 3, 9, 7]
merge_sort_bottomup(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 60)
    t = [random.randint(0, 40) for _ in range(n)]
    ref = sorted(t)
    merge_sort_bottomup(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")
print("재귀 깊이: 0 (재귀 없음)")

### 17. 🟢 [정리] 정렬 6종 총정리

이번 주로 교재의 주요 정렬을 거의 다 봤어. 표를 직접 채워봐. (외우지 말고 **왜 그런지** 떠올리면서)

| 정렬 | 평균 | 최악 | 추가 메모리 | 안정? | 배운 날 | 한 줄 특징 |
|---|---|---|---|---|---|---|
| 버블 | O(n²) | O(n²) | O(1) | ✅ | 18일 | 인접 교환 |
| 선택 | ① | ② | O(1) | ③ | 19일 | 최솟값을 앞으로 |
| 삽입 | ④ | ⑤ | O(1) | ⑥ | 19일 | 거의 정렬된 데이터에 강함 |
| 셸 | ⑦ | ⑧ | O(1) | ⑨ | 20일 | 삽입 정렬 + 간격 h |
| 퀵 | ⑩ | ⑪ | ⑫ | ⑬ | 21~22일 | 나누고 각각 정렬 |
| **병합** | ⑭ | ⑮ | ⑯ | ⑰ | **오늘** | 각각 정렬하고 합침 |

**심화 질문**
- 퀵 정렬은 최악 O(n²)인데 병합은 **최악도 O(n log n)** 이야. 그런데도 실무에서 퀵을 많이 쓰는 이유는? (15번 실측을 근거로)
- 병합 정렬의 유일한 단점은 **추가 메모리 O(n)** 이야. 어떤 상황에서 이게 치명적일까?
- 파이썬 `sorted()`의 **Timsort**는 병합 + 삽입 하이브리드야. 이번 주에 배운 것 중 **어떤 성질들**을 조합한 걸까? (안정성, 거의 정렬된 데이터 강점, 최악 보장 — 각각 어느 정렬에서 왔지?)

*(표를 채운 뒤 아래 셀로 실측 확인)*

In [ ]:
# 이번 주 정렬 총출동 — 같은 데이터로 실측
def selection_sort(a):
    n = len(a)
    for i in range(n - 1):
        m = i
        for j in range(i + 1, n):
            if a[j] < a[m]: m = j
        a[i], a[m] = a[m], a[i]

def insertion_sort(a):
    for i in range(1, len(a)):
        j = i; tmp = a[i]
        while j > 0 and a[j-1] > tmp:
            a[j] = a[j-1]; j -= 1
        a[j] = tmp

def shell_sort(a):
    n = len(a); h = 1
    while h < n // 9: h = h * 3 + 1
    while h > 0:
        for i in range(h, n):
            j = i - h; tmp = a[i]
            while j >= 0 and a[j] > tmp:
                a[j+h] = a[j]; j -= h
            a[j+h] = tmp
        h //= 3

random.seed(9)
SMALL = [random.randint(0, 10**6) for _ in range(3000)]
print(f"[n = {len(SMALL)}, 랜덤 입력]")
for name, f in (("선택", selection_sort), ("삽입", insertion_sort), ("셸", shell_sort),
                ("퀵", quick_sort), ("병합", merge_sort), ("내장", lambda x: x.sort())):
    a = SMALL[:]
    t = time.perf_counter(); f(a); el = time.perf_counter() - t
    print(f"  {name:4s}: {el:7.4f}s  {'OK' if a == sorted(SMALL) else 'X'}")

NEARLY = sorted(SMALL); 
for _ in range(30):                      # 거의 정렬된 데이터
    i = random.randrange(len(NEARLY)); j = random.randrange(len(NEARLY))
    NEARLY[i], NEARLY[j] = NEARLY[j], NEARLY[i]
print(f"\n[n = {len(NEARLY)}, 거의 정렬된 입력]")
for name, f in (("선택", selection_sort), ("삽입", insertion_sort), ("셸", shell_sort),
                ("퀵", quick_sort), ("병합", merge_sort), ("내장", lambda x: x.sort())):
    a = NEARLY[:]
    t = time.perf_counter(); f(a); el = time.perf_counter() - t
    print(f"  {name:4s}: {el:7.4f}s")

---
---

# ✅ 정답 & 해설

> ⚠️ **먼저 다 풀고 내려와.** 특히 9번과 12번은 답을 보면 재미가 사라져.

---

## 🔁 Remind

### R-1
```
a = [2,5,5,9], b = [1,5,7]
1: 2<=1? 거짓 → b에서 1        c=[1]
2: 2<=5? 참   → a에서 2        c=[1,2]
3: 5<=5? 참   → a에서 5 ★      c=[1,2,5]
4: 5<=5? 참   → a에서 5 ★      c=[1,2,5,5]
5: 9<=5? 거짓 → b에서 5        c=[1,2,5,5,5]
6: 9<=7? 거짓 → b에서 7        c=[1,2,5,5,5,7]
   b 소진 → ③번 while이 a의 9를 복사
```
- ★ 값이 같을 때 **a가 먼저** 나가는 이유는 `<=` 의 **`=` 한 글자**. 이게 안정성의 전부야 (14번에서 다시).
- 마지막에 실행되는 건 **③번** (`while pa < na`) — b가 먼저 비었으니까.

### R-2
- 퀵은 피벗이 나쁘면 한쪽으로 치우쳐서 **깊이가 n까지** 갈 수 있어 (21일차 9번: 깊이 5 vs 125).
- 병합은 `center = (left+right)//2` 로 **항상 정확히 절반**. 데이터를 **보지 않고** 나눠.
- 그래서 재귀 깊이는 **입력에 전혀 무관하게 항상 ⌈log₂n⌉ 근처**. 15번 실측에서 랜덤/정렬/역순 전부 깊이 18로 똑같이 나와.

> 🔑 **퀵은 "나누기가 어렵고 합치기가 공짜", 병합은 "나누기가 공짜고 합치기가 일"** 이야. 분할 정복의 두 얼굴.

---

## 🧩 PART 1 해설

### 1. 세 개의 커서

| pc | 비교 | 저장 | (pa, pb) |
|---|---|---|---|
| 0 | `2 <= 1` 거짓 | 1 | (0, 1) |
| 1 | `2 <= 2` **참** | 2 | (1, 1) |
| 2 | `4 <= 2` 거짓 | 2 | (1, 2) |
| 3 | `4 <= 3` 거짓 | 3 | (1, 3) |
| 4 | `4 <= 4` **참** | 4 | (2, 3) |

- 값이 같으면 **a가 먼저** (`<=` 덕분).
- **`pc`만 매 반복 반드시 1 증가**해. `pa`와 `pb`는 **둘 중 하나만** 증가하지. → 그래서 `pc`는 항상 `pa + pb`와 같아.

---

### 2. 왜 while이 세 개인가

- ①이 끝났다 = `pa == na` **또는** `pb == nb` = 둘 중 **적어도 하나가 다 소진됐다**.
- ②와 ③은 **동시에 실행될 수 없어.** ②가 돌려면 a가 남아 있어야 하는데, 그건 b가 비었다는 뜻이고, 그럼 ③의 조건은 처음부터 거짓이거든.
- 둘 다 지우면 **길이가 다른 순간** 바로 깨져. 예: `a=[1,2,3]`, `b=[9]` → ①은 b를 못 꺼내고 a만 3개 꺼내다 끝나서 `c=[1,2,3,None]`.
- `c`에 저장되는 횟수는 정확히 `na + nb` = **n번**. 각 원소가 딱 한 번씩만 이동하니까 → **O(n)**.

> 🔑 병합이 O(n)이라는 게 병합 정렬 O(n log n)의 **절반**이야. 나머지 절반(log n)은 6번에서.

---

### 3. ②번 while 삭제

- `a=[1,2,9]`, `b=[3,4]` → **조용히 틀린다.** `[1,2,3,4,None]` — 9가 사라지고 None이 남아.
- `a=[1,2]`, `b=[3,4,9]` → **멀쩡히 통과.** a가 먼저 비고 ③번이 살아 있으니까.

**같은 버그인데 입력에 따라 드러나기도, 숨기도 해.** 22일차 10번(`pr -= 1`)과 14번(`j > 0`)에서 본 그 패턴이야. 테스트 케이스를 `a`가 긴 것/`b`가 긴 것 **둘 다** 넣어야 잡히는 버그.

---

### 4. 병합하는 세 가지 방법 — 그리고 교재와 다른 실측

| 방법 | 시간 복잡도 | 정렬 안 돼 있어도 OK? |
|---|---|---|
| `merge_sorted_list` | ① **O(n)** | ② ❌ (반드시 정렬되어 있어야) |
| `sorted(a + b)` | ③ **O(n log n)** | ④ ✅ |
| `heapq.merge(a, b)` | ⑤ **O(n log k)** (k = 배열 개수, 2개면 사실상 O(n)) | ⑥ ❌ |

**실측 (각 20만 개)**

```
직접 병합(파이썬 루프) : 0.09 ~ 0.15s
sorted(a + b)          : 0.05 ~ 0.12s   ← 더 빠르다?!
heapq.merge(a, b)      : 0.10 ~ 0.11s
```
*(정확한 값은 실행 환경마다 달라. 순위가 뒤집힐 수도 있으니 직접 몇 번 돌려봐.)*

교재는 `sorted(a+b)`가 **"속도가 빠르지 않다"** 고 했는데 실측은 반대야. 왜?

- **알고리즘 복잡도로는 교재가 맞아.** O(n) vs O(n log n)이니까.
- 그런데 `sorted()`는 **C로 구현**되어 있고, 우리가 짠 병합은 **파이썬 인터프리터**가 한 줄씩 해석해. 파이썬 루프 한 번이 C 루프보다 수십 배 느려서, log n(≈19) 배수를 **상수 차이로 눌러버린 거야.**
- 게다가 Timsort는 **이미 정렬된 구간(run)을 감지**해서 `a+b`가 두 개의 정렬된 덩어리라는 걸 알아채. 실제로는 O(n)에 가깝게 동작해.

> 🔑 **복잡도는 "n이 충분히 클 때"의 이야기고, 실제 속도는 상수와 구현 언어가 지배한다.** 20만 개로도 부족했다는 뜻이야. 이번 주 42746 문제에서 직접 짠 퀵이 `cmp_to_key`보다 빨랐던 것과 정확히 반대 방향의 교훈.

---

### 5. `heapq.merge()`는 제너레이터

- `list()` 없이 출력하면 `<generator object merge at 0x...>` 가 나와. **아직 아무것도 계산 안 한 상태**야.
- 두 번 순회하면 **1회차 13개, 2회차 0개.** 제너레이터는 한 번 흘러가면 끝이야 (물이 흐르는 것과 같아, 다시 담아두지 않으면 사라져).
- **장점이 되는 상황**: 결과 전체를 메모리에 안 올려도 돼. 실측을 보면 200만 개 병합에서 앞 5개만 뽑을 때 `heapq.merge`는 0.00003s, `sorted(a+b)[:5]`는 0.408s — **약 1만 배 차이**야. `sorted`는 200만 개를 전부 정렬한 다음에야 앞 5개를 줄 수 있거든.
- 실무 예: 거대한 로그 파일 10개를 시간순으로 병합해서 스트리밍 처리할 때.

> 🔑 3일차 `range()`가 리스트가 아니었던 것, 4일차 `reversed()`가 이터레이터였던 것과 **같은 계보**야. 파이썬은 "필요할 때 계산한다(lazy)"를 좋아해.

---

### 6. 쪼개기

- ① `center = (0+11)//2 = 5`
- ② 앞부분 `a[0]~a[5]` (6개)  ③ 뒷부분 `a[6]~a[11]` (6개)
- 12 → 6 → 3 → 2 → 1 이므로 **4단계**. 일반적으로 **④ log₂n** 단계.

**교재 280p 3줄**
```
1. ⑤ 배열의 앞부분을 병합 정렬로 정렬합니다
2. ⑥ 배열의 뒷부분을 병합 정렬로 정렬합니다
3. ⑦ 배열의 앞부분과 뒷부분을 병합합니다
```

**퀵 정렬과의 결정적 차이**
- 퀵: `분할(정렬 작업) → 재귀 → 재귀` — **정렬이 재귀 앞에** 있어. 합치는 일이 없어(이미 제자리).
- 병합: `재귀 → 재귀 → 병합(정렬 작업)` — **정렬이 재귀 뒤에** 있어.
- 재귀 호출은 **둘 다 2번**. 구조는 쌍둥이인데 **일하는 시점**이 정반대야.

> 🔑 이걸 각각 **top-down 분할 정복**의 두 유형이라고 해. "나누면서 일하기" vs "합치면서 일하기".

---

## 🔬 PART 2 해설

### 7. `if left < right:`

- 거짓이 되는 건 `left == right`(원소 1개) 또는 `left > right`(원소 0개). 1개짜리는 이미 정렬 완료.
- `<=` 로 바꾸면: `left == right == 3`일 때 `center = 3`, 그럼 `_merge_sort(a, 3, 3)` 를 **자기 자신과 똑같은 인자로** 다시 호출 → **무한 재귀**. 실행 결과처럼 `RecursionError`.
- 22일차 3번에서는 조건을 빼자 **스택이 폭발**했고, 여기서는 **호출 스택이 폭발**해. R-2에서 말한 대로 **같은 것의 두 얼굴**이야.

---

### 8. `buff`를 쓰는 3단계

**ⓐ** ① `buff = [1, 3, 4, 6, 11, 12]`, ② `p = 6` (= `center - left + 1`)

**ⓑ** 처음 4번:
```
k=0 ← buff[0]=1    (1 <= 2)
k=1 ← a[6]=2       (3 <= 2 거짓)
k=2 ← buff[1]=3    (3 <= 3, = 이라서 buff 우선 ★)
k=3 ← a[7]=3
```
끝나는 두 경우: ③ `i > right` (뒷부분 소진) 또는 ④ `j >= p` (buff 소진)

**ⓒ 🔥 왜 "a의 뒷부분이 남았을 때"를 위한 while이 없을까**

뒷부분이 남았다는 건 **`buff`가 먼저 비었다**는 뜻이야. 그런데 `a`의 뒷부분 원소들은 **처음부터 `a`의 그 자리에 있었고**, 9번에서 볼 `k ≤ i` 불변식 때문에 **아직 덮어써지지 않았어.** 즉 **이미 정답 위치에 있어서 아무것도 할 필요가 없다.**

반대로 `buff`에 남은 건 `a`에서 **떠온 복사본**이라, 되돌려놓지 않으면 영원히 사라져. 그래서 ⓒ만 필요한 거야.

> 🔑 이 비대칭이 이 알고리즘의 우아한 지점이야. 22일차 17번 `merge_sorted_list`는 while이 3개였는데(a용, b용 둘 다 필요), 여기선 2개면 충분한 이유.

---

### 9. 🔥 오늘의 핵심 — `k ≤ i` 불변식

```
k = left + j + (i - center - 1)
j <= p = center - left + 1  이므로
k <= left + (center - left + 1) + (i - center - 1) = ① i
```

- ① **`i`** → 결론 **`k ≤ ② i`**
- 즉 **쓰는 위치가 읽는 위치를 절대 추월하지 않아.** 최악의 경우 딱 같은 칸을 쓰는데, 그건 그 값을 이미 읽고 난 뒤야.
- 실측: 랜덤 500회 × 모든 병합 단계에서 **위반 0회**.

**앞부분은 왜 복사가 필수인가**: `k`는 `left`에서 시작하고 앞부분 원본도 `left`에 있어. 복사 안 하면 **첫 번째 쓰기가 곧바로 `a[left]`를 덮어써.** 뒷부분과 달리 여유 공간이 전혀 없어.

**22일차 14번과의 차이** 🔥
| | 22일차 `while j > 0` | 오늘 `k ≤ i` |
|---|---|---|
| 안전한가 | 예 | 예 |
| 왜 안전한가 | **호출자**의 성질에 의존 (우연) | **알고리즘 자체**가 보장 (설계) |
| 다른 데서 재사용하면 | 조용히 깨짐 | 그대로 안전 |
| 함수가 자기 계약을 지키나 | ❌ | ✅ |

> 🔑 **불변식은 "우연히 성립하는 것"과 "설계로 보장되는 것"이 하늘과 땅 차이다.** 좋은 알고리즘은 후자를 증명할 수 있어.

---

### 10. `and j < p` 제거

- `j == p` = `buff`를 다 꺼냈다 → `buff[p]` 이후는 `[None]*n`으로 만들 때의 **`None`이 그대로**.
- `None <= 정수` 비교 → `TypeError: '<=' not supported between instances of 'NoneType' and 'int'`
- 다행히 **시끄럽게 실패**해. 3번(조용히 틀림)보다 훨씬 나은 실패야.
- `i <= right`만 지우면: `a[right+1]` 너머를 읽어서 **다른 구간의 데이터를 침범**하거나, 마지막 구간에선 `IndexError`. 이쪽이 더 위험해 — 침범해도 에러가 안 날 수 있거든.

---

### 11. 병합 정렬 완성

```python
if left < right:                     # ①
    center = (left + right) // 2     # ②
    _ms(a, left, center)             # ③
    _ms(a, center + 1, right)        # ④
    while i <= center:               # ⑤
    while i <= right and j < p:      # ⑥
        if buff[j] <= a[i]:          # ⑦  ← <= 여야 안정 정렬!
    while j < p:                      # ⑧
```
출력: `[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]`, 랜덤 500회 실패 0회

---

### 12. 🔥 예측이 틀리는 문제

**실측 (n = 50,000)**

| | 시간 | 피크 메모리 |
|---|---|---|
| buff 1회 생성 (교재) | (거의 동일) | 약 **392 KB** |
| buff 매번 생성 | (거의 동일) | 약 **196 KB** |

**둘 다 예상과 다를 거야.**

- **시간**: 거의 차이 없음. 파이썬의 리스트 할당이 생각만큼 비싸지 않고, 진짜 비용은 **병합 루프 자체**거든. 할당 횟수(n번)는 총 연산(n log n)에 비하면 미미해.
- **메모리**: 매번 생성이 **오히려 절반**! 교재 방식은 `n`짜리를 처음부터 끝까지 붙들고 있지만, 매번 생성 방식은 재귀가 끝나면 즉시 해제되어 동시에 살아 있는 게 최대 `n/2` 수준이야.

**그럼 교재는 왜 밖에 뒀을까?**
- 코드가 **한 줄 줄어들고**, `(center - left + 1)` 같은 **크기 계산 실수 가능성이 사라져.**
- C 같은 언어에서는 할당(`malloc`)이 훨씬 비싸서 이 최적화가 실제로 유효해. 교재의 관습이 그쪽에서 온 거야.
- 그리고 최악의 순간(메모리 부족)에 **정렬 도중에 할당 실패**하는 위험을 피할 수 있어.

> 🔑 **"최적화처럼 보이는 것"이 이 언어/이 규모에서도 최적화인지는 재봐야 안다.** 22일차 10번(`pr -= 1`)에서는 실측이 교재 편을 들어줬고, 오늘은 반대로 나왔어. **둘 다 재봤기 때문에 알 수 있는 거야.**

---

### 13. 클로저와 `del`

**(1) 스코프**
- 파이썬은 이름을 **정의 시점이 아니라 실행 시점에** 찾아. `_merge_sort`가 실제로 호출될 때는 이미 `buff = [None] * n`이 실행된 뒤라 문제없어.
- 이걸 **클로저**라고 해. 안쪽 함수가 바깥 함수의 지역 변수를 들여다보는 구조. 실행 셀의 `demo()`가 그대로 보여줘 — `msg`를 `inner` 정의 **뒤에** 만들어도 잘 동작하지.
- 어디에도 없는 이름이면? 그때 비로소 `NameError`.

**(2) `del buff`**
- `del`은 **이름표를 떼는 것**이지 메모리를 즉시 해제하는 명령이 아니야. 실행 셀에서 `del lst` 후에도 `alias`로 리스트가 멀쩡히 살아 있는 게 그 증거.
- 참조가 0이 되면 그때 파이썬이 알아서 회수해. `merge_sort`가 곧 끝나면 `buff`도 어차피 사라지니 **실질적 차이는 거의 없어.**
- 교재가 쓴 이유는 **"이 작업용 배열은 여기서 역할이 끝났다"는 의도 표시**에 가까워. 문서화 목적의 코드지.

> 🔑 3일차 "가변/불변", 4일차 "call by object reference"의 연장선이야. **이름과 객체는 별개**라는 파이썬의 근본 규칙.

---

## 📐 PART 3 해설

### 14. 안정성

**실측**
```
입력       : ['a','b','c','d','e','f']  값 [3,1,3,1,2,3]
<= (교재)  : ['b','d','e','a','c','f']  값 [1,1,2,3,3,3]   ✅ 안정
<  (바꿈)  : ['d','b','e','f','c','a']  값 [1,1,2,3,3,3]   ❌ 순서 뒤집힘
파이썬 sorted(): ['b','d','e','a','c','f']                 ✅ 동일
```

- ① 안정이면 `b, d, e, a, c, f` (1은 b→d, 3은 a→c→f 순서 유지)
- **왜 `buff`가 앞부분이라는 게 중요한가**: 같은 값일 때 `buff[j] <= a[i]`가 참이 되어 **`buff` 쪽을 먼저** 꺼내. `buff`는 배열의 **앞부분**이니까, 결국 **원래 앞에 있던 원소가 앞에 남는다.** 안정성의 정의 그 자체야.
- **퀵과의 대조**: 퀵은 `a[pl]`과 `a[pr]`을 교환하는데, 이 둘이 **멀리 떨어져 있어.** 같은 값 사이를 뛰어넘어 자리를 바꿔버리니 원래 순서가 보존될 수 없어. 교재 285p의 **"서로 떨어져 있는 원소를 교환하는 것이 아니므로"** 가 정확히 이 말이야.
- 병합은 **교환을 아예 안 해.** 순서대로 꺼내서 순서대로 쓸 뿐이지.

---

### 15. O(n log n)과 퀵과의 대결

**재귀 깊이 (n = 100,000, log₂n ≈ 16.6)**
```
랜덤 입력      : 18
이미 정렬된 입력: 18     ← 완전히 동일
역순 입력      : 18
```
**입력에 전혀 무관.** R-2의 예측이 맞았지. 퀵 정렬이었다면 피벗에 따라 요동쳤을 거야.

**속도 (n = 100,000)**

| 입력 | 병합 | 퀵 | 내장 |
|---|---|---|---|
| 랜덤 | 0.292s | **0.157s** | 0.020s |
| 이미 정렬됨 | 0.210s | **0.095s** | 0.001s |

| 항목 | 병합 | 퀵 |
|---|---|---|
| ① 랜덤 속도 | 느림 (약 1.9배) | 빠름 |
| ② 정렬된 입력 | 안정적 | 가운데 피벗이라 여전히 빠름 |
| ③ 추가 메모리 | **O(n)** | **O(log n)** (재귀 스택만) |
| ④ 최악 시간 | **O(n log n) 보장** | O(n²) |

**퀵이 빠른 이유**: 제자리(in-place) 교환이라 **메모리 이동이 적고**, 캐시에 잘 맞아. 병합은 매 단계마다 `buff`로 복사하고 다시 쓰는 왕복이 있어.

---

### 16. 상향식 병합 정렬

```python
for lo in range(0, n, 2 * width):        # ① 한 쌍(앞+뒤)씩 건너뛴다
    hi = min(lo + 2 * width, n)          # ②
    if buff[j] <= a[i]:                  # ③ 안정성 유지
width *= 2                               # ④
```
출력: `[1, 2, 3, 4, 5, 6, 7, 8, 9]`, 랜덤 500회 실패 0회, **재귀 깊이 0**

**퀵의 비재귀화(22일차)와 비교**

| | 퀵 비재귀 | 병합 비재귀 |
|---|---|---|
| 스택 필요? | ✅ 필수 | ❌ 불필요 |
| 왜? | 나눌 위치가 **데이터에 따라 결정**되어 미리 알 수 없음 | 나눌 위치가 **항상 절반**이라 미리 계산 가능 |
| 진행 방향 | 위 → 아래 | **아래 → 위** |

> 🔑 **"데이터를 봐야 아는 것"과 "미리 아는 것"의 차이**가 자료구조 필요 여부를 갈라. 이게 오늘 배운 것 중 가장 실전적인 통찰이야.

💡 실무 노트: 파이썬의 Timsort가 바로 이 **상향식** 방식이야. 이미 정렬된 구간(run)을 찾아서 아래에서부터 병합해 올라가.

---

### 17. 정렬 6종 총정리

| 정렬 | 평균 | 최악 | 추가 메모리 | 안정? | 한 줄 |
|---|---|---|---|---|---|
| 버블 | O(n²) | O(n²) | O(1) | ✅ | 인접 교환 |
| 선택 | ① O(n²) | ② O(n²) | O(1) | ③ ❌ | 최솟값을 앞으로 |
| 삽입 | ④ O(n²) | ⑤ O(n²) | O(1) | ⑥ ✅ | 거의 정렬된 데이터에 강함 |
| 셸 | ⑦ ~O(n^1.25) | ⑧ O(n²) | O(1) | ⑨ ❌ | 삽입 + 간격 h |
| 퀵 | ⑩ O(n log n) | ⑪ O(n²) | ⑫ O(log n) | ⑬ ❌ | 나누고 각각 정렬 |
| **병합** | ⑭ O(n log n) | ⑮ **O(n log n)** | ⑯ **O(n)** | ⑰ **✅** | 각각 정렬하고 합침 |

**실측 (n = 3,000)**

| 정렬 | 랜덤 | 거의 정렬됨 |
|---|---|---|
| 선택 | 0.214s | 0.196s ← **거의 안 줄어듦** |
| 삽입 | 0.231s | **0.006s** ← **38배 빨라짐** |
| 셸 | 0.007s | 0.003s |
| 퀵 | **0.003s** | 0.002s |
| 병합 | 0.006s | 0.005s |
| 내장 | 0.0005s | 0.0001s |

- **선택 정렬만 거의 안 줄어드는 이유**: 데이터가 어떻든 항상 남은 구간 전체를 훑어 최솟값을 찾으니까. 19일차에 배운 그대로.
- **삽입 정렬의 38배**: 이미 정렬되어 있으면 `while` 조건이 즉시 거짓 → 사실상 O(n).

**심화 답**
- **퀵을 많이 쓰는 이유**: 최악 O(n²)는 **가운데 피벗 + sort3**로 실질적으로 회피 가능하고(22일차), 평균 속도가 병합보다 1.9배 빠르며 **추가 메모리가 O(log n)** 뿐이야.
- **O(n) 메모리가 치명적인 상황**: 임베디드/모바일처럼 RAM이 빠듯할 때, 또는 정렬 대상이 메모리의 절반 이상을 차지할 때.
- **Timsort의 조합**:
  - **안정성** ← 병합 정렬 (오늘)
  - **거의 정렬된 데이터에서 폭발적 성능** ← 삽입 정렬 (19일차)
  - **최악 O(n log n) 보장** ← 병합 정렬 (오늘)
  - **작은 구간은 삽입 정렬로 전환** ← 22일차 개선 퀵과 같은 아이디어

> 🔑 **Timsort는 이번 주에 배운 것들의 총합이야.** 우리가 19~23일차에 하나씩 배운 성질들을 실제 언어 설계자가 어떻게 조합했는지 보이지?

---

## 📌 핵심 3줄 요약

1. **병합 정렬은 "나누기가 공짜, 합치기가 일"이다.** 퀵은 반대야. `center = (left+right)//2`로 **데이터를 안 보고** 나누기 때문에 재귀 깊이가 입력에 무관하게 항상 log n — 실측 랜덤/정렬/역순 전부 깊이 18로 동일했어.
2. **앞부분만 `buff`에 복사해도 되는 이유는 `k ≤ i` 불변식이다.** 쓰는 위치가 읽는 위치를 절대 추월하지 않아서 뒷부분은 제자리에 둬도 안전해. 22일차 14번의 "우연히 안전"과 달리, 이건 **부등식으로 증명되는 설계**야.
3. **안정성은 `buff`가 앞부분이라는 사실 + `<=` 한 글자에서 나온다.** 그래서 병합 정렬은 유일하게 **안정 + 최악 O(n log n)** 을 동시에 만족하고, 파이썬 `sorted()`가 이 계열을 고른 이유가 됐어.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 6, 7, 8, 17번)**: 전원 필수
  - **1번 커서 추적**과 **8번 buff 3단계**가 오늘의 기본기. 이 둘을 손으로 못 그리면 11번 빈칸이 안 채워져
  - **17번 총정리표**는 코테/면접 직전에 다시 볼 것
- 🟡 **(R-2, 3, 4, 5, 10, 11, 14, 15번)**: 팀 목표선
  - **4번 실측 반전**(교재와 반대 결과)이 오늘의 재미 포인트 🔥
  - **11번 빈칸**은 반드시 손으로. 오늘 배운 걸 전부 쓰는 문제야
- 🔴 **(9, 12, 13, 16번)**: 도전
  - **9번이 오늘 최고 난도** 🔥🔥 — `k ≤ i` 부등식을 직접 세워보면 "왜 이 알고리즘이 옳은가"를 증명한 셈이야
  - **12번**은 예측이 틀리는 게 정상. 틀린 뒤에 "그럼 왜?"를 파는 게 목적
  - **16번 상향식**은 22일차 비재귀 퀵과 대조하면서 볼 것
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 15번)

## 🔗 오늘 회수된 개념들

- **22일차 17번 `merge_sorted_list`** → 오늘 병합 정렬의 부품으로 그대로 투입 (R-1, 1~3번)
- **22일차 18번 안정성 예고** → 완성된 병합 정렬로 실증 (14번)
- **22일차 3번 기저 조건 / 스택 폭발** → `if left < right` 없으면 재귀 폭발 (7번)
- **22일차 14번 "우연히 안전한 코드"** → 오늘의 `k ≤ i`는 "설계로 안전한 코드" (9번)
- **22일차 10번 `pr -= 1` 실측** → 오늘 12번은 실측이 교재와 **반대로** 나옴
- **22일차 비재귀 퀵(스택)** → 병합은 스택이 왜 필요 없는지 (16번)
- **21일차 퀵 정렬 피벗/깊이** → 병합은 깊이가 입력 무관 (R-2, 15번)
- **19일차 삽입 정렬** → 거의 정렬된 데이터에서 38배 (17번), Timsort의 재료
- **3~4일차 이름 vs 객체, 이터레이터** → 클로저, `del`, 제너레이터 (5, 13번)

---

> **다음 진도 (24일차)**: 06-8 **힙 정렬** — 오늘 잠깐 등장한 `heapq`의 정체가 밝혀져.
> 힙은 **"항상 최댓값이 맨 위에 있는 트리"** 야. 선택 정렬(19일차)이 매번 최댓값을 O(n)에 찾았다면, 힙은 O(log n)에 찾아. **선택 정렬의 진화형**이라고 생각하면 돼.